In [37]:
import datetime as dt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import requests
import re
import pickle 
import pickle as pk
import keras
import tensorflow as tf
import os

from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

from keras import layers
from keras.layers import RepeatVector
from keras.callbacks import ModelCheckpoint, EarlyStopping, TensorBoard
from keras.models import Sequential, Model, load_model
from keras.layers import Input, Dense, Activation, Flatten, Dropout
from keras.layers import SimpleRNN, GRU
from keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Input, LSTM, RepeatVector, TimeDistributed
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import IsolationForest

from torchdiffeq import odeint
import torchcde  

from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, accuracy_score

import plotly.graph_objs as go

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Pickle Save object
def save_obj(obj, filename):
    filename = 'CSV data/' + filename + '.pkl'
    with open(filename, 'wb') as outp:  
        pickle.dump(obj, outp, pickle.HIGHEST_PROTOCOL)

# Pickle Read Object
def read_obj(filename):
    
    filename = 'CSV data/' + filename + '.pkl'
    with open(filename, 'rb') as inp: 
        return pickle.load(inp)

In [3]:
### Visualization 2020 to 2023

df = pd.read_csv("./CSV data/GT/GT_final.csv")
df.head()

start_date = '2020-01-01 00:00:00'
num_rows = len(df)
date_range = pd.date_range(start=start_date, periods=num_rows, freq='H')

df['date'] = date_range
df.to_csv("./CSV data/GT/GT_final_date.csv", index = False)


/var/folders/gc/ccxrs945073cqwbghj93h9wr0000gn/T/ipykernel_30765/832067052.py:8: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  date_range = pd.date_range(start=start_date, periods=num_rows, freq='H')


In [35]:
df = pd.read_csv("./CSV data/GT/GT_final_date.csv")

In [ ]:
### spike 처리 

date = df['date']
df = df[df.iloc[:, :-1] < 300]
df = df[df.iloc[:, :-1] > -300]
df['date'] = date

### Neural ODE

In [38]:
def train_enhanced_anomaly_detector(model, train_loader, optimizer, num_epochs, device):
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        
        for x_batch, time_steps in train_loader:
            x_batch = x_batch.to(device)
            time_steps = time_steps.to(device)
            
            # 1. 정상 데이터 처리
            encoded_normal = model.encoder(x_batch, time_steps)
            z0_normal = encoded_normal[:, -1, :]
            reconstructed_normal = model.decoder(z0_normal, time_steps)
            
            # 재구성 손실
            recon_loss = F.mse_loss(reconstructed_normal, x_batch)
            
            # 2. 인공 이상치 생성
            noise_level = min(0. + epoch * 0.01, 0.5)  # 점진적 노이즈 증가
            noise_mask = (torch.rand_like(x_batch[:, :, 0:1]) < 0.2).float()
            noise = torch.randn_like(x_batch) * noise_level * noise_mask
            noisy_x = x_batch + noise
            
            # 이상치 데이터 처리
            encoded_anomaly = model.encoder(noisy_x, time_steps)
            z0_anomaly = encoded_anomaly[:, -1, :]
            reconstructed_anomaly = model.decoder(z0_anomaly, time_steps)
            
            # 3. 특징 대조 손실
            normal_features = encoded_normal.mean(dim=1)
            anomaly_features = encoded_anomaly.mean(dim=1)
            contrast_loss = feature_contrastive_loss(normal_features, anomaly_features, margin=2.0)
            
            # 4. 시간적 일관성 손실
            normal_scores = model(x_batch, time_steps)
            anomaly_scores = model(noisy_x, time_steps)
            temporal_loss = temporal_consistency_loss(normal_scores) + temporal_consistency_loss(anomaly_scores)
            
            # 5. 분류 손실
            class_loss = F.binary_cross_entropy(normal_scores, torch.zeros_like(normal_scores)) + \
                         F.binary_cross_entropy(anomaly_scores, torch.ones_like(anomaly_scores) * 0.9)
            
            # 최종 손실 계산
            loss = 0.4 * recon_loss + 0.3 * contrast_loss + 0.1 * temporal_loss + 0.2 * class_loss
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)  # 그래디언트 클리핑
            optimizer.step()
            
            total_loss += loss.item()
        
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(train_loader):.6f}")
        

In [80]:
# 통합된 이상 탐지 평가 함수
def evaluate_anomaly_detector(
    model, 
    test_data=None, 
    test_labels=None, 
    test_loader=None,
    device=None, 
    threshold=None,
    threshold_method='auto',
    threshold_percentile=95,
    return_scores=False,
    verbose=True
):
    """
    통합된 이상 탐지 모델 평가 함수
    - DataLoader와 직접 데이터 입력 모두 지원
    - 다양한 임계값 설정 방법 제공
    - 풍부한 평가 지표 계산
    
    Args:
        model: 학습된 모델
        test_data: 테스트 데이터 (직접 입력)
        test_labels: 테스트 라벨 (직접 입력)
        test_loader: DataLoader (test_data/test_labels와 함께 사용 불가)
        device: 디바이스 설정
        threshold: 수동 임계값 설정
        threshold_method: 'percentile', 'normal_based', 'auto' 중 선택
        threshold_percentile: percentile 방법 사용시 백분위수
        return_scores: 이상 점수 반환 여부
        verbose: 결과 출력 여부
    
    Returns:
        dict: 평가 지표들
    """
    
    # 디바이스 설정
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model = model.to(device)
    model.eval()
    
    # 입력 검증
    if test_loader is not None and (test_data is not None or test_labels is not None):
        raise ValueError("test_loader와 test_data/test_labels를 동시에 사용할 수 없습니다.")
    
    if test_loader is None and (test_data is None or test_labels is None):
        raise ValueError("test_loader 또는 test_data/test_labels를 제공해야 합니다.")
    
    # 이상 점수 계산
    anomaly_scores = []
    all_labels = []
    
    with torch.no_grad():
        if test_loader is not None:
            # DataLoader 사용 경우
            for batch_data in test_loader:
                if len(batch_data) == 3:  # (x, times, mask)
                    x_batch, times_batch, mask_batch = batch_data
                    x_batch = x_batch.to(device)
                    times_batch = times_batch.to(device)
                    mask_batch = mask_batch.to(device)
                    
                    scores = model.anomaly_score(x_batch, times_batch, mask_batch)
                    anomaly_scores.extend(scores.cpu().numpy())
                else:
                    raise ValueError("DataLoader의 배치 형태가 예상과 다릅니다.")
            
            # test_loader에서 라벨 추출 (별도로 제공되어야 함)
            if test_labels is None:
                raise ValueError("test_loader 사용시 test_labels를 별도로 제공해야 합니다.")
            all_labels = test_labels
            
        else:
            # 직접 데이터 입력 경우
            if isinstance(test_data, np.ndarray):
                test_data = torch.tensor(test_data, dtype=torch.float32)
            
            # 차원 처리
            if test_data.dim() == 2:  # (T, D) → (1, T, D)
                test_data = test_data.unsqueeze(0)
            
            # 시간 벡터 생성
            T = test_data.shape[1]
            timestamps = torch.linspace(0, 1, T).unsqueeze(0).expand(test_data.shape[0], -1).to(device)
            
            # 마스크 생성
            mask = torch.ones_like(timestamps)
            
            # 데이터를 디바이스로 이동
            test_data = test_data.to(device)
            
            # 이상 점수 계산
            scores = model.anomaly_score(test_data, timestamps, mask)
            anomaly_scores = scores.cpu().numpy()
            all_labels = test_labels
    
    # 점수와 라벨 형태 맞추기
    anomaly_scores = np.array(anomaly_scores)
    all_labels = np.array(all_labels)
    
    # 차원 조정
    if all_labels.ndim == 1 and len(anomaly_scores.shape) > 1 and anomaly_scores.shape[0] == 1:
        # 단일 시퀀스 (1, T)와 벡터 라벨 (T,)
        scores_flat = anomaly_scores.squeeze()
        labels_flat = all_labels
    elif all_labels.ndim == 2:
        # 다중 시퀀스와 2D 라벨
        scores_flat = anomaly_scores.reshape(-1)
        labels_flat = all_labels.reshape(-1)
    elif len(anomaly_scores.shape) > 1:
        # 다중 시퀀스와 1D 라벨 - 윈도우별 최대값 사용
        scores_flat = np.max(anomaly_scores, axis=1) if anomaly_scores.ndim > 1 else anomaly_scores
        labels_flat = all_labels
    else:
        scores_flat = anomaly_scores
        labels_flat = all_labels
    
    # 임계값 설정
    if threshold is None:
        if threshold_method == 'percentile':
            threshold = np.percentile(scores_flat, threshold_percentile)
        elif threshold_method == 'normal_based':
            # 정상 데이터 기반 임계값
            normal_scores = scores_flat[labels_flat == 0]
            if len(normal_scores) > 0:
                threshold = np.mean(normal_scores) + 2 * np.std(normal_scores)
            else:
                threshold = np.percentile(scores_flat, 95)
        elif threshold_method == 'auto':
            # 자동 선택: 정상 데이터가 충분하면 normal_based, 아니면 percentile
            normal_scores = scores_flat[labels_flat == 0]
            if len(normal_scores) > len(scores_flat) * 0.7:  # 정상 데이터가 70% 이상
                threshold = np.mean(normal_scores) + 2 * np.std(normal_scores)
            else:
                threshold = np.percentile(scores_flat, threshold_percentile)
        else:
            raise ValueError(f"지원하지 않는 threshold_method: {threshold_method}")
    
    # 예측 생성
    predictions = (scores_flat > threshold).astype(int)
    
    # 성능 지표 계산
    metrics = {}
    
    try:
        # 기본 분류 지표
        metrics['accuracy'] = accuracy_score(labels_flat, predictions)
        metrics['precision'] = precision_score(labels_flat, predictions, average='binary', zero_division=0)
        metrics['recall'] = recall_score(labels_flat, predictions, average='binary', zero_division=0)
        metrics['f1'] = f1_score(labels_flat, predictions, average='binary', zero_division=0)
        
        # ROC-AUC (레이블이 다양한 경우에만)
        if len(np.unique(labels_flat)) > 1:
            metrics['auc'] = roc_auc_score(labels_flat, scores_flat)
        else:
            metrics['auc'] = None
            
    except Exception as e:
        if verbose:
            print(f"지표 계산 중 오류 발생: {e}")
        metrics['accuracy'] = 0.0
        metrics['precision'] = 0.0
        metrics['recall'] = 0.0
        metrics['f1'] = 0.0
        metrics['auc'] = None
    
    # 추가 정보
    metrics['threshold'] = threshold
    metrics['threshold_method'] = threshold_method
    metrics['num_samples'] = len(labels_flat)
    metrics['num_anomalies'] = np.sum(labels_flat)
    metrics['num_predicted_anomalies'] = np.sum(predictions)
    
    # 이상 점수 반환 옵션
    if return_scores:
        metrics['anomaly_scores'] = scores_flat
        metrics['predictions'] = predictions
        metrics['labels'] = labels_flat
    
    # 결과 출력
    if verbose:
        print(f"=== 이상 탐지 평가 결과 ===")
        print(f"Threshold ({threshold_method}): {threshold:.4f}")
        print(f"Accuracy: {metrics['accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall: {metrics['recall']:.4f}")
        print(f"F1 Score: {metrics['f1']:.4f}")
        if metrics['auc'] is not None:
            print(f"AUC: {metrics['auc']:.4f}")
        print(f"Samples: {metrics['num_samples']}, "
              f"True Anomalies: {metrics['num_anomalies']}, "
              f"Predicted: {metrics['num_predicted_anomalies']}")
    
    return metrics

In [81]:
def plot_spike_detection_for_variable(x, time_steps, anomaly_scores, var_idx=0, threshold=None, sample_idx=0, percentile=95):
    """
    x: (batch, seq_len, input_dim) - 입력 시계열
    time_steps: (batch, seq_len) - 시간 정보
    anomaly_scores: (batch, seq_len) - 모델 출력 anomaly score
    var_idx: int - 시각화할 변수 인덱스 (ex. 0~127)
    sample_idx: int - 배치 중 어떤 시계열을 볼지 (기본값: 0)
    threshold: float - anomaly 판단 기준값. 없으면 percentile 기반 계산
    percentile: float - threshold를 anomaly_scores의 상위 몇 %로 지정할지
    """

    # Tensor → NumPy
    x_np = x.detach().cpu().numpy()
    time_np = time_steps.detach().cpu().numpy()
    score_np = anomaly_scores.detach().cpu().numpy()

    # 하나의 시계열 추출
    values = x_np[sample_idx, :, var_idx]
    times = time_np[sample_idx]
    scores = score_np[sample_idx]

    # threshold 계산 (percentile 기반 or 수동 입력)
    if threshold is None:
        threshold = np.percentile(scores, percentile)
    
    # spike 위치
    spikes = scores > threshold

    # 시각화
    fig, ax1 = plt.subplots(figsize=(14, 5))

    # 실제 값 plot
    ax1.set_title(f"Spike Detection for Variable {var_idx}")
    ax1.plot(times, values, label=f'Variable {var_idx} Value', color='blue', marker='o')
    ax1.set_xlabel("Time")
    ax1.set_ylabel("Variable Value", color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')

    # spike 점 찍기
    ax1.scatter(times[spikes], values[spikes], color='red', label='Detected Spikes', zorder=5)

    # anomaly score plot (우측 축)
    ax2 = ax1.twinx()
    ax2.plot(times, scores, color='orange', linestyle='--', label='Anomaly Score')
    ax2.axhline(threshold, color='gray', linestyle=':', label='Threshold')
    ax2.set_ylabel("Anomaly Score", color='orange')
    ax2.tick_params(axis='y', labelcolor='orange')

    # 범례 정리
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # 간단 출력
    print("Total spikes detected:", spikes.sum())
    print("Threshold used:", threshold)


In [102]:
def train_irregular_ode(model, data_dict, num_epochs=30):
    """
    Train and evaluate the irregular ODE model
    
    Args:
        model: Model to train
        data_dict: Dictionary containing train_loader, test_loader, and labels
        num_epochs: Number of epochs to train
    
    Returns:
        model: Trained model
        history: Training history
        metrics: Evaluation metrics
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    # Optimizer setup
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )
    
    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=True
    )
    
    # Training
    model.train()
    history = []
    
    for epoch in range(num_epochs):
        epoch_loss = 0
        epoch_recon_loss = 0
        epoch_kl_loss = 0
        epoch_flow_loss = 0
        
        for batch_idx, (x_batch, times_batch, mask_batch) in enumerate(data_dict['train_loader']):
            x_batch = x_batch.to(device)
            times_batch = times_batch.to(device)
            mask_batch = mask_batch.to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(x_batch, times_batch, mask_batch)
            losses = outputs["losses"]
            
            # Backward pass
            losses["total_loss"].backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            # Accumulate losses
            epoch_loss += losses["total_loss"].item()
            epoch_recon_loss += losses["recon_loss"].item()
            epoch_kl_loss += losses["kl_loss"].item()
            epoch_flow_loss += losses["flow_loss"].item()
        
        # Average losses per epoch
        n_batches = len(data_dict['train_loader'])
        avg_loss = epoch_loss / n_batches
        avg_recon_loss = epoch_recon_loss / n_batches
        avg_kl_loss = epoch_kl_loss / n_batches
        avg_flow_loss = epoch_flow_loss / n_batches
        
        # Learning rate adjustment
        scheduler.step(avg_loss)
        
        print(f"Epoch {epoch+1}/{num_epochs}, "
              f"Loss: {avg_loss:.4f}, "
              f"Recon: {avg_recon_loss:.4f}, "
              f"KL: {avg_kl_loss:.4f}, "
              f"Flow: {avg_flow_loss:.4f}")
        
        history.append({
            'epoch': epoch + 1,
            'total_loss': avg_loss,
            'recon_loss': avg_recon_loss,
            'kl_loss': avg_kl_loss,
            'flow_loss': avg_flow_loss
        })
    
    # Evaluation
    model.eval()
    anomaly_scores = []
    
    with torch.no_grad():
        for x_batch, times_batch, mask_batch in data_dict['test_loader']:
            x_batch = x_batch.to(device)
            times_batch = times_batch.to(device)
            mask_batch = mask_batch.to(device)
            
            # Calculate anomaly scores
            scores = model.anomaly_score(x_batch, times_batch, mask_batch)
            anomaly_scores.extend(scores)
    
    anomaly_scores = np.array(anomaly_scores)
    
    # Convert labels to numpy and check dimensions
    if isinstance(data_dict['lab_tst'], torch.Tensor):
        test_labels = data_dict['lab_tst'].cpu().numpy()
    else:
        test_labels = np.array(data_dict['lab_tst'])
    
    # Adjust dimensions
    if test_labels.ndim > 1:
        test_labels = test_labels.flatten()
    if anomaly_scores.ndim > 1:
        anomaly_scores = anomaly_scores.flatten()
    
    # Match sizes (if lengths differ)
    min_len = min(len(test_labels), len(anomaly_scores))
    test_labels = test_labels[:min_len]
    anomaly_scores = anomaly_scores[:min_len]
    
    # Debug information
    print(f"Test labels shape: {test_labels.shape}")
    print(f"Anomaly scores shape: {anomaly_scores.shape}")
    print(f"Test labels unique values: {np.unique(test_labels)}")
    print(f"Anomaly scores range: [{anomaly_scores.min():.4f}, {anomaly_scores.max():.4f}]")
    
    # Set threshold (90th percentile)
    threshold = np.percentile(anomaly_scores, 90)
    predictions = (anomaly_scores > threshold).astype(int)
    
    # Calculate performance metrics
    metrics = {}
    
    try:
        # Calculate AUC (check binary classification)
        if len(np.unique(test_labels)) == 2:
            metrics['auc'] = roc_auc_score(test_labels, anomaly_scores)
        else:
            print(f"Warning: Not binary classification. Unique labels: {np.unique(test_labels)}")
            metrics['auc'] = None
        
        # Other metrics
        metrics['f1'] = f1_score(test_labels, predictions, average='binary')
        metrics['precision'] = precision_score(test_labels, predictions, average='binary', zero_division=0)
        metrics['recall'] = recall_score(test_labels, predictions, average='binary', zero_division=0)
        metrics['accuracy'] = accuracy_score(test_labels, predictions)
        
        print(f"AUC: {metrics['auc']:.4f}" if metrics['auc'] else "AUC: N/A")
        print(f"F1 Score: {metrics['f1']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall: {metrics['recall']:.4f}")
        print(f"Accuracy: {metrics['accuracy']:.4f}")
        
    except Exception as e:
        print(f"Error in metrics calculation: {e}")
        metrics = {
            'auc': None,
            'f1': 0.0,
            'precision': 0.0,
            'recall': 0.0,
            'accuracy': 0.0
        }
    
    # Additional information
    metrics.update({
        'threshold': threshold,
        'anomaly_scores': anomaly_scores,
        'predictions': predictions,
        'test_labels': test_labels
    })
    
    return model, history, metrics

In [83]:
class ProbabilisticDecoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_dim):
        super().__init__()
        self.output_dim = output_dim
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim * 2)
        )
        
        # 가중치 초기화
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
    
    def forward(self, z):
        # 디코딩
        out = self.decoder(z)
        
        # 출력을 mean과 log variance로 분리
        mean, logvar = torch.chunk(out, 2, dim=-1)
        
        # log variance 클램핑
        logvar = torch.clamp(logvar, min=-20, max=20)
        
        return mean, logvar

In [84]:
# ODE-RNN/GRU-D + Time2Vec + Conditional Flow (최종)

class Time2Vec(nn.Module):
    def __init__(self, out_dim, num_frequencies=4):
        super().__init__()
        self.out_dim = out_dim
        self.num_frequencies = num_frequencies
        
        # Linear component
        self.linear = nn.Linear(1, 1)
        
        # Periodic components with multiple frequencies
        self.frequencies = nn.Parameter(torch.randn(num_frequencies) * 0.1)
        
        # 출력 차원을 정확히 맞추기 위해 계산
        periodic_dim = out_dim - 1  # 전체 차원에서 linear 차원(1)을 뺌
        self.periodic = nn.Linear(1, periodic_dim)
        
        print(f"Time2Vec initialized with:")
        print(f"- Total output dimension: {out_dim}")
        print(f"- Linear component dimension: 1")
        print(f"- Periodic component dimension: {periodic_dim}")
        
    def forward(self, t):
        # Linear component
        v1 = self.linear(t)  # (B, T, 1)
        
        # Periodic component
        periodic = torch.sin(self.frequencies[0] * self.periodic(t))  # (B, T, out_dim-1)
        
        # Concatenate and verify dimension
        output = torch.cat([v1, periodic], dim=-1)  # (B, T, out_dim)
        assert output.size(-1) == self.out_dim, \
            f"Expected output dimension {self.out_dim}, got {output.size(-1)}"
        
        return output

class ODERNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, time_emb_dim=8):
        super().__init__()
        self.time2vec = Time2Vec(time_emb_dim)
        
        # 입력 차원 계산 및 확인
        self.input_dim = input_dim
        self.time_emb_dim = time_emb_dim
        self.total_input_dim = input_dim + time_emb_dim
        
        print(f"Initializing ODERNNEncoder with:")
        print(f"- Input dimension: {input_dim}")
        print(f"- Time embedding dimension: {time_emb_dim}")
        print(f"- Total input dimension: {self.total_input_dim}")
        
        self.rnn = nn.GRU(self.total_input_dim, hidden_dim, batch_first=True)
        self.mu = nn.Linear(hidden_dim, latent_dim)
        self.logvar = nn.Linear(hidden_dim, latent_dim)
        
    def forward(self, x, times, mask=None):
        # 입력 차원 확인
        print(f"Input shape: {x.shape}")
        
        # 시간 임베딩
        t_emb = self.time2vec(times.unsqueeze(-1))  # (B, T, time_emb_dim)
        print(f"Time embedding shape: {t_emb.shape}")
        
        # 입력과 시간 임베딩 결합
        x_cat = torch.cat([x, t_emb], dim=-1)  # (B, T, input_dim + time_emb_dim)
        print(f"Concatenated shape: {x_cat.shape}")
        
        # 차원 검증
        assert x_cat.size(-1) == self.total_input_dim, \
            f"Expected input dimension {self.total_input_dim}, got {x_cat.size(-1)}"
        
        # RNN 처리
        h, _ = self.rnn(x_cat)  # (B, T, hidden_dim)
        h_last = h[:, -1]  # (B, hidden_dim)
        
        # 잠재 공간으로 매핑
        mu = self.mu(h_last)  # (B, latent_dim)
        logvar = self.logvar(h_last)  # (B, latent_dim)
        
        # 잠재 변수 샘플링
        z0 = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        
        return z0, mu, logvar
    
class IrregularTimeSeriesAnomalyDetector(nn.Module):
    def __init__(self, input_dim, latent_dim=32, hidden_dim=64, flow_layers=4, time_emb_dim=8, beta=0.1):
        super().__init__()
        
        # 차원이 짝수인지 확인
        if latent_dim % 2 != 0:
            latent_dim += 1
            print(f"Adjusted latent_dim to {latent_dim} to ensure it's even.")
        
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim
        self.time_emb_dim = time_emb_dim
        self.beta = beta
        self.temperature = 0.1  
        self.anomaly_weight = 5.0 
        
        # 인코더 (ODE-RNN + Time2Vec)
        self.encoder = ODERNNEncoder(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            latent_dim=latent_dim,
            time_emb_dim=time_emb_dim
        )
        
        # ODE 함수 (Time2Vec 포함)
        self.odefunc = ODEFunc(
            latent_dim=latent_dim,
            hidden_dim=hidden_dim,
            time_emb_dim=time_emb_dim
        )
        
        # 디코더
        self.decoder = ProbabilisticDecoder(
            latent_dim=latent_dim,
            output_dim=input_dim,
            hidden_dim=hidden_dim
        )
        
        # Flow 모델
        self.flow = SimpleFlow(dim=latent_dim, n_layers=flow_layers)
        
        # 사전 확률 분포 파라미터
        self.register_buffer('prior_mean', torch.zeros(latent_dim))
        self.register_buffer('prior_logvar', torch.zeros(latent_dim))
        
    def encode(self, x, times, mask=None):
        return self.encoder(x, times, mask)
        
    def decode(self, z0, times):
        if times.dim() > 1:
            times = times[0]
        
        times, indices = torch.sort(times)
        
        zt = odeint(
            self.odefunc,
            z0,
            times,
            method='dopri5',
            rtol=1e-3,
            atol=1e-4
        )
        
        zt = zt.permute(1, 0, 2)
        mu, logvar = self.decoder(zt)
        
        return mu, logvar
        
    def temporal_contrastive_loss(self, z_normal, z_augmented, times, temperature=0.1):
        """
        시간적 연속성을 고려한 대조 학습
        """
        # L2 정규화
        z_normal = F.normalize(z_normal, dim=-1)
        z_augmented = F.normalize(z_augmented, dim=-1)
        
        # 시간 차이에 따른 가중치 계산
        time_diff = torch.abs(times[:, 1:] - times[:, :-1])
        time_weights = torch.exp(-time_diff)  # 시간이 가까울수록 가중치가 큼
        
        # 유사도 계산
        similarity = torch.mm(z_normal, z_augmented.t()) / temperature
        
        # 시간적 연속성을 고려한 loss
        temporal_loss = 0
        for i in range(z_normal.size(0)-1):
            if i < time_weights.size(1):
                weight = time_weights[0, i]
                temporal_loss += weight * F.cross_entropy(
                    similarity[i:i+1], 
                    torch.tensor([i], device=z_normal.device)
                )
        
        return temporal_loss / (z_normal.size(0) - 1)

    def compute_loss(self, x, times, recon_mu, recon_logvar, mu, logvar, z_flow, log_det, mask=None):
        device = x.device
        batch_size = x.shape[0]
        
        # 기존 loss들
        if mask is not None:
            mask_expanded = mask.unsqueeze(-1).expand_as(x)
            recon_error = (recon_mu - x) ** 2
            recon_loss = (recon_error * mask_expanded).sum() / (mask_expanded.sum() + 1e-8)
        else:
            recon_loss = F.mse_loss(recon_mu, x, reduction='mean')
        
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / batch_size
        
        if z_flow is not None:
            flow_loss = 0.5 * torch.sum(z_flow**2) / batch_size
        else:
            flow_loss = torch.tensor(0.0, device=device)
        
        # 데이터 증강 및 시간적 대조 학습
        if self.training and batch_size > 1:
            # 노이즈 증강
            noise_factor = 0.01
            x_augmented = x + torch.randn_like(x) * noise_factor
            
            # 증강된 데이터의 잠재 표현
            z0_aug, _, _ = self.encode(x_augmented, times, mask)
            
            # 시간적 대조 학습 loss
            contrast_loss = self.temporal_contrastive_loss(mu, z0_aug, times)
        else:
            contrast_loss = torch.tensor(0.0, device=device)
        
        anomaly_loss = self.compute_anomaly_aware_loss(x, recon_mu, recon_logvar)

        total_loss = (
            recon_loss + 
            self.beta * kl_loss + 
            0.01 * flow_loss + 
            0.2 * contrast_loss +
            0.1 * anomaly_loss 
        )
        
        return {
            'total_loss': total_loss,
            'recon_loss': recon_loss,
            'kl_loss': kl_loss,
            'flow_loss': flow_loss,
            'contrast_loss': contrast_loss,
            'anomaly_loss': anomaly_loss
        }

    def compute_anomaly_aware_loss(self, x, recon_mu, recon_logvar):
        """이상치 감지를 강화하기 위한 손실 함수"""
        # 재구성 오차 계산
        recon_error = torch.abs(x - recon_mu)
        
        # 동적 임계값 계산 (평균 + 표준편차)
        mean_error = torch.mean(recon_error, dim=(0, 1))
        std_error = torch.std(recon_error, dim=(0, 1))
        threshold = mean_error + 2 * std_error
        
        # 이상치 가중치 계산
        anomaly_weights = torch.where(
            recon_error > threshold.unsqueeze(0).unsqueeze(0),
            torch.ones_like(recon_error) * self.anomaly_weight,
            torch.ones_like(recon_error)
        )
        
        # 가중치가 적용된 재구성 손실
        weighted_loss = torch.mean(recon_error * anomaly_weights)
        
        return weighted_loss
    
    def forward(self, x, times, mask=None):
        if times.dim() == 1:
            times = times.unsqueeze(0).expand(x.size(0), -1)
        
        unique_times = torch.linspace(0, 1, times.size(1)).to(times.device)
        
        z0, mu, logvar = self.encode(x, times, mask)
        z_flow, log_det = self.flow(z0)
        recon_mu, recon_logvar = self.decode(z0, unique_times)
        
        losses = self.compute_loss(x, times, recon_mu, recon_logvar, mu, logvar, z_flow, log_det, mask)
        
        return {
            'z0': z0,
            'mu': mu,
            'logvar': logvar,
            'z_flow': z_flow,
            'log_det': log_det,
            'recon_mu': recon_mu,
            'recon_logvar': recon_logvar,
            'losses': losses
        }
        
    def anomaly_score(self, x, times, mask=None):
        self.eval()
        
        with torch.no_grad():
            outputs = self.forward(x, times, mask)
            
            # 1. 개선된 재구성 오차 (시계열 특성 고려)
            recon_error = torch.abs(x - outputs['recon_mu'])
            std = torch.exp(0.5 * outputs['recon_logvar']) + 1e-6
            
            # Point-wise anomaly score with temporal smoothing
            pointwise_scores = recon_error / std
            
            # Temporal smoothing (이동평균)
            kernel_size = min(5, x.shape[1])
            if kernel_size > 1:
                padding = kernel_size // 2
                pointwise_scores = F.avg_pool1d(
                    pointwise_scores.transpose(1, 2), 
                    kernel_size, stride=1, padding=padding
                ).transpose(1, 2)
            
            if mask is not None:
                mask_expanded = mask.unsqueeze(-1)
                pointwise_scores = pointwise_scores * mask_expanded
            
            # Feature-wise aggregation with temporal attention
            time_diff = torch.abs(times[:, 1:] - times[:, :-1])
            time_weights = torch.exp(-time_diff)  # 시간이 가까울수록 가중치가 큼
            time_weights = F.pad(time_weights, (0, 1), value=1.0)
            
            # 5. 특성별 점수 계산
            feature_scores = pointwise_scores * time_weights.unsqueeze(-1)
            
            # 6. 시퀀스 레벨 통계 계산
            max_scores = torch.max(feature_scores, dim=1)[0]  # (B, F)
            mean_scores = torch.mean(feature_scores, dim=1)   # (B, F)
            std_scores = torch.std(feature_scores, dim=1)     # (B, F)
            
            # 7. KL 발산 점수 계산
            kl_scores = 0.5 * torch.sum(
                outputs['mu'].pow(2) + torch.exp(outputs['logvar']) - outputs['logvar'] - 1,
                dim=-1
            ).unsqueeze(-1)

            # 3. Flow-based density score
            flow_scores = 0.5 * torch.sum(outputs['z_flow']**2, dim=-1).unsqueeze(-1)  # (B, 1)
            
            dynamic_threshold = mean_scores + 2 * std_scores

            final_scores = (
                0.35 * max_scores +     # 최대 재구성 오차
                0.25 * mean_scores +    # 평균 재구성 오차
                0.15 * std_scores +     # 재구성 오차 변동성
                0.15 * kl_scores.expand(-1, self.input_dim) +  # KL 발산
                0.10 * flow_scores.expand(-1, self.input_dim)  # Flow 밀도
            )
            
            anomaly_mask = final_scores > dynamic_threshold
            final_scores = torch.where(
                anomaly_mask,
                final_scores * self.anomaly_weight,
                final_scores
            )
            
            # 12. 시퀀스 레벨 점수로 변환
            sequence_scores = torch.mean(final_scores, dim=-1)  # (B,)
            
            # 13. 점수 정규화
            sequence_scores = (sequence_scores - sequence_scores.min()) / (sequence_scores.max() - sequence_scores.min() + 1e-6)
            
            return sequence_scores.cpu().numpy()

class ODERNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, time_emb_dim=8):
        super().__init__()
        self.time2vec = Time2Vec(time_emb_dim)
        self.rnn = nn.GRU(input_dim + time_emb_dim, hidden_dim, batch_first=True)
        self.mu = nn.Linear(hidden_dim, latent_dim)
        self.logvar = nn.Linear(hidden_dim, latent_dim)
    def forward(self, x, times, mask=None):
        t_emb = self.time2vec(times.unsqueeze(-1))  # (B, T, time_emb_dim)
        x_cat = torch.cat([x, t_emb], dim=-1)
        h, _ = self.rnn(x_cat)
        h_last = h[:, -1]
        mu = self.mu(h_last)
        logvar = self.logvar(h_last)
        z0 = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        return z0, mu, logvar

class ODEFunc(nn.Module):
    def __init__(self, latent_dim, hidden_dim, time_emb_dim=8):
        super().__init__()
        self.time2vec = Time2Vec(time_emb_dim)
        
        # 시간 임베딩과 상태를 결합한 네트워크
        self.net = nn.Sequential(
            nn.Linear(latent_dim + time_emb_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, latent_dim)
        )
        
    def forward(self, t, z):
        # 시간 임베딩
        t_tensor = t.reshape(1, 1).repeat(z.size(0), 1)
        t_emb = self.time2vec(t_tensor)
        
        # 상태와 시간 임베딩 결합
        z_t = torch.cat([z, t_emb], dim=1)
        
        # 속도장 계산
        dz_dt = self.net(z_t)
        return dz_dt

class ProbabilisticDecoder(nn.Module):
    def __init__(self, latent_dim, output_dim, hidden_dim):
        super().__init__()
        # 더 깊은 네트워크와 정규화 추가
        self.decoder_net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),  # Layer Normalization 추가
            nn.ReLU(),
            nn.Dropout(0.1),  # Dropout 추가
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
        )
        
        # 출력 레이어 분리 및 초기화 개선
        self.mu_net = nn.Linear(hidden_dim, output_dim)
        self.logvar_net = nn.Linear(hidden_dim, output_dim)
        
        # 가중치 초기화
        self._init_weights()

    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, z):
        h = self.decoder_net(z)
        mu = self.mu_net(h)
        logvar = self.logvar_net(h)
        # 분산의 범위를 더 넓게 설정
        logvar = torch.clamp(logvar, -20, 2)
        return mu, logvar
    

class SimpleFlow(nn.Module):
    def __init__(self, dim, n_layers=2):
        super().__init__()
        if dim % 2 != 0:
            raise ValueError(f"Dimension must be even, got {dim}")
            
        self.layers = nn.ModuleList()
        for _ in range(n_layers):
            self.layers.append(AffineCoupling(dim))
            
    def forward(self, z):
        log_det = torch.zeros(z.size(0), device=z.device)
        for layer in self.layers:
            z, ldj = layer(z)
            log_det += ldj
        return z, log_det

class AffineCoupling(nn.Module):
    def __init__(self, dim, hidden_dim=64):
        super().__init__()
        self.dim = dim
        assert dim % 2 == 0, "dim must be even"
        
        self.net = nn.Sequential(
            nn.Linear(dim // 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, dim)
        )
    
    def forward(self, x, reverse=False):
        x1, x2 = torch.chunk(x, 2, dim=1)
        h = self.net(x1)
        shift, scale = torch.chunk(h, 2, dim=1)
        scale = torch.tanh(scale)
        
        if not reverse:
            y2 = x2 * torch.exp(scale) + shift
        else:
            y2 = (x2 - shift) * torch.exp(-scale)
            
        y = torch.cat([x1, y2], dim=1)
        log_det = scale.sum(dim=1)
        
        if reverse:
            log_det = -log_det
            
        return y, log_det

### Open dataset

1. SMD

In [47]:
### SMD 
import kagglehub
import shutil
import os
from datasets.SMD.preprocess_SMD import SMD_Dataset  

# Download latest version
path = kagglehub.dataset_download("mgusat/smd-onmiad")

print("Path to dataset files:", path)

# 현재 경로 (KaggleHub 캐시)
source_path = "/Users/jieun/.cache/kagglehub/datasets/mgusat/smd-onmiad/versions/1"
# 원하는 경로 (예: ./data/SMD)
destination_path = "./datasets/SMD"

os.makedirs(destination_path, exist_ok=True)

for item in os.listdir(source_path):
    s = os.path.join(source_path, item)
    d = os.path.join(destination_path, item)
    if os.path.isdir(s):
        shutil.copytree(s, d, dirs_exist_ok=True)
    else:
        shutil.copy2(s, d)

print("✅ SMD 데이터 복사 완료:", destination_path)

Path to dataset files: /Users/jieun/.cache/kagglehub/datasets/mgusat/smd-onmiad/versions/1
✅ SMD 데이터 복사 완료: ./datasets/SMD


In [48]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [70]:
def prepare_smd_data(x_trn, x_tst, lab_tst, batch_size=32):
    # 데이터 형태 확인
    print(f"Original data shapes:")
    print(f"x_trn: {x_trn.shape}, x_tst: {x_tst.shape}, lab_tst: {lab_tst.shape}")
    
    # 데이터 정규화 (Min-Max 스케일링)
    def min_max_scaler(data):
        data_reshaped = data.reshape(-1, data.shape[-1])
        min_vals = np.min(data_reshaped, axis=0)
        max_vals = np.max(data_reshaped, axis=0)
        scaled = (data_reshaped - min_vals) / (max_vals - min_vals + 1e-8)
        return scaled.reshape(data.shape), min_vals, max_vals
    
    # 학습 데이터 정규화
    x_trn_norm, min_vals, max_vals = min_max_scaler(x_trn)
    x_tst_norm = (x_tst.reshape(-1, x_tst.shape[-1]) - min_vals) / (max_vals - min_vals + 1e-8)
    x_tst_norm = x_tst_norm.reshape(x_tst.shape)
    
    # 데이터가 2D인 경우 3D로 변환 (시계열 윈도우 생성)
    if len(x_trn.shape) == 2:  # (samples, features)
        print("Converting 2D data to 3D windows...")
        window_size = min(50, x_trn.shape[0] // 10)  # 적절한 윈도우 크기 설정
        
        # 윈도우 생성 함수
        def create_windows(data, window_size, stride=1):
            windows = []
            for i in range(0, len(data) - window_size + 1, stride):
                windows.append(data[i:i + window_size])
            return np.array(windows)
        
        # 학습 데이터 윈도우 생성
        x_trn_windowed = create_windows(x_trn_norm, window_size)
        
        # 테스트 데이터 윈도우 생성
        x_tst_windowed = create_windows(x_tst_norm, window_size)
        
        # 라벨도 윈도우에 맞게 조정 (윈도우 내 최대값 사용)
        if len(lab_tst.shape) == 1:
            lab_tst_windowed = []
            for i in range(0, len(lab_tst) - window_size + 1):
                lab_tst_windowed.append(np.max(lab_tst[i:i + window_size]))
            lab_tst_windowed = np.array(lab_tst_windowed)
        else:
            lab_tst_windowed = create_windows(lab_tst, window_size)
            lab_tst_windowed = np.max(lab_tst_windowed, axis=1)
        
        # 업데이트된 데이터 사용
        x_trn_norm = x_trn_windowed
        x_tst_norm = x_tst_windowed
        lab_tst = lab_tst_windowed
        
        print(f"After windowing:")
        print(f"x_trn: {x_trn_norm.shape}, x_tst: {x_tst_norm.shape}, lab_tst: {lab_tst.shape}")
    
    # 텐서 변환
    x_trn_tensor = torch.FloatTensor(x_trn_norm)
    x_tst_tensor = torch.FloatTensor(x_tst_norm)
    lab_tst_tensor = torch.FloatTensor(lab_tst)
    
    def create_time_vector(seq_length):
        return torch.linspace(0, 1, seq_length)
    
    # 시간 벡터 생성 (3D 데이터 기준)
    if len(x_trn_tensor.shape) == 3:  # (batch, seq_len, features)
        train_times = create_time_vector(x_trn_tensor.shape[1])
        train_times = train_times.unsqueeze(0).expand(x_trn_tensor.shape[0], -1)
        
        test_times = create_time_vector(x_tst_tensor.shape[1])
        test_times = test_times.unsqueeze(0).expand(x_tst_tensor.shape[0], -1)
        
        # 마스크 생성 - 올바른 차원으로
        train_mask = torch.ones_like(x_trn_tensor[:, :, 0])  # (batch, seq_len)
        test_mask = torch.ones_like(x_tst_tensor[:, :, 0])   # (batch, seq_len)
        
    else:  # 2D 데이터인 경우
        raise ValueError(f"Data should be 3D after processing, but got shape {x_trn_tensor.shape}")
    
    # 데이터셋 생성
    train_dataset = TensorDataset(x_trn_tensor, train_times, train_mask)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_dataset = TensorDataset(x_tst_tensor, test_times, test_mask)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    scaler_params = {
        'min_vals': min_vals,
        'max_vals': max_vals
    }
    
    return {
        'train_loader': train_loader,
        "test_loader": test_loader,
        'x_tst': x_tst_tensor,
        'train_times': train_times,
        'test_times': test_times,
        'train_mask': train_mask,
        'test_mask': test_mask,
        'lab_tst': lab_tst_tensor,
        'scaler_params': scaler_params 
    }
    
def train_and_evaluate_smd(model, data_dict, num_epochs=30):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    # 옵티마이저 설정
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )
    
    # 학습률 스케줄러
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=True
    )
    
    # 학습
    model, history = train_irregular_ode(
        model,
        data_dict['train_loader'],
        optimizer,
        scheduler,
        num_epochs=num_epochs
    )
    
    # 평가
    model.eval()
    anomaly_scores = []
    
    with torch.no_grad():
        for x_batch, times_batch, mask_batch in data_dict['test_loader']:
            x_batch = x_batch.to(device)
            times_batch = times_batch.to(device)
            mask_batch = mask_batch.to(device)
            
            # 이상 점수 계산
            scores = model.anomaly_score(x_batch, times_batch, mask_batch)
            anomaly_scores.extend(scores)
    
    anomaly_scores = np.array(anomaly_scores)
    
    # 라벨을 numpy로 변환하고 차원 확인
    if isinstance(data_dict['lab_tst'], torch.Tensor):
        test_labels = data_dict['lab_tst'].cpu().numpy()
    else:
        test_labels = np.array(data_dict['lab_tst'])
    
    # 차원 맞추기
    if test_labels.ndim > 1:
        test_labels = test_labels.flatten()
    if anomaly_scores.ndim > 1:
        anomaly_scores = anomaly_scores.flatten()
    
    # 크기 맞추기 (길이가 다른 경우)
    min_len = min(len(test_labels), len(anomaly_scores))
    test_labels = test_labels[:min_len]
    anomaly_scores = anomaly_scores[:min_len]
    
    # 디버깅 정보 출력
    print(f"Test labels shape: {test_labels.shape}")
    print(f"Anomaly scores shape: {anomaly_scores.shape}")
    print(f"Test labels unique values: {np.unique(test_labels)}")
    print(f"Anomaly scores range: [{anomaly_scores.min():.4f}, {anomaly_scores.max():.4f}]")
    
    # 임계값 설정 (95번째 백분위수)
    threshold = np.percentile(anomaly_scores, 90)
    predictions = (anomaly_scores > threshold).astype(int)
    
    # 성능 지표 계산
    metrics = {}
    
    try:
        # AUC 계산 (이진 분류 확인)
        if len(np.unique(test_labels)) == 2:
            metrics['auc'] = roc_auc_score(test_labels, anomaly_scores)
        else:
            print(f"Warning: Not binary classification. Unique labels: {np.unique(test_labels)}")
            metrics['auc'] = None
        
        # 다른 지표들
        metrics['f1'] = f1_score(test_labels, predictions, average='binary')
        metrics['precision'] = precision_score(test_labels, predictions, average='binary', zero_division=0)
        metrics['recall'] = recall_score(test_labels, predictions, average='binary', zero_division=0)
        metrics['accuracy'] = accuracy_score(test_labels, predictions)
        
        print(f"AUC: {metrics['auc']:.4f}" if metrics['auc'] else "AUC: N/A")
        print(f"F1 Score: {metrics['f1']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall: {metrics['recall']:.4f}")
        print(f"Accuracy: {metrics['accuracy']:.4f}")
        
    except Exception as e:
        print(f"Error in metrics calculation: {e}")
        metrics = {
            'auc': None,
            'f1': 0.0,
            'precision': 0.0,
            'recall': 0.0,
            'accuracy': 0.0
        }
    
    # 추가 정보
    metrics.update({
        'threshold': threshold,
        'anomaly_scores': anomaly_scores,
        'predictions': predictions,
        'test_labels': test_labels
    })
    
    return model, history, metrics

In [71]:
class Params:
    def __init__(self, dl, stride, tst_stride, train_method='train_all_entities', clamp_max=None, clamp_min=None, entity_id=None, downsample=None):
        self.dl = dl
        self.stride = stride
        self.tst_stride = tst_stride
        self.train_method = train_method
        self.clamp_max = clamp_max
        self.clamp_min = clamp_min
        self.entity_id = entity_id
        self.downsample = downsample

In [72]:
def main():
    # SMD 데이터 로드
    with open('./datasets/SMD/ServerMachineDataset/SMD.pk', 'rb') as f:
        data = pickle.load(f)
    
    # 첫 번째 엔티티 사용
    entity_idx = 0
    x_trn = data['x_trn'][entity_idx]
    x_tst = data['x_tst'][entity_idx]
    lab_tst = data['lab_tst'][entity_idx]
    
    # 윈도우 레벨 라벨로 변환
    if lab_tst.ndim == 2:
        lab_tst = np.max(lab_tst, axis=1)
    
    print(f"x_trn shape: {x_trn.shape}")
    print(f"x_tst shape: {x_tst.shape}")
    print(f"lab_tst shape: {lab_tst.shape}")
    print(f"Anomaly ratio: {np.mean(lab_tst):.3f}")
    
    # 데이터 전처리
    data_dict = prepare_smd_data(x_trn, x_tst, lab_tst)
    
    # 모델 초기화
    model = IrregularTimeSeriesAnomalyDetector(
        input_dim=x_trn.shape[-1],
        latent_dim=32,
        hidden_dim=64,
        flow_layers=6,
        time_emb_dim=8,
        beta=0.1
    )
    
    # 학습 및 평가
    model, history, metrics = train_and_evaluate_smd(model, data_dict)
    
    # history 내용 확인 및 시각화
    print("History keys:", list(history.keys()) if isinstance(history, dict) else "Not a dictionary")
    print("History type:", type(history))
    
    # 안전한 방식으로 손실 그래프 그리기
    plt.figure(figsize=(15, 5))
    
    try:
        if isinstance(history, dict):
            # 가능한 키들을 확인하고 사용
            loss_keys = ['loss', 'total_loss']  # 가능한 키들
            recon_keys = ['recon_loss', 'reconstruction_loss']
            kl_keys = ['kl_loss', 'kl_divergence']
            flow_keys = ['flow_loss']
            
            # 총 손실 그래프
            for key in loss_keys:
                if key in history:
                    plt.plot(history[key], label=f'Total Loss ({key})')
                    break
            
            # 재구성 손실 그래프
            for key in recon_keys:
                if key in history:
                    plt.plot(history[key], label=f'Reconstruction Loss ({key})')
                    break
            
            # KL 손실 그래프
            for key in kl_keys:
                if key in history:
                    plt.plot(history[key], label=f'KL Loss ({key})')
                    break
            
            for key in flow_keys:
                if key in history:
                    plt.plot(history[key], label=f'Flow Loss ({key})')
                    break
            
            plt.title('Training Loss History')
            plt.xlabel('Epoch')
            plt.ylabel('Loss')
            plt.legend()
            plt.grid(True)
            plt.show()
            
        else:
            print("History is not a dictionary. Cannot plot loss curves.")
            print("History content:", history)
            
    except Exception as e:
        print(f"Error plotting history: {e}")
        print("Available history keys:", list(history.keys()) if isinstance(history, dict) else "Not available")
    
    if isinstance(metrics, dict):
        print(f"\n=== Final Results ===")
        print(f"AUC: {metrics.get('auc', 'N/A'):.4f}" if metrics.get('auc') else "AUC: N/A")
        print(f"F1 Score: {metrics.get('f1', 'N/A'):.4f}" if metrics.get('f1') else "F1: N/A")
        print(f"Precision: {metrics.get('precision', 'N/A'):.4f}" if metrics.get('precision') else "Precision: N/A")
        print(f"Recall: {metrics.get('recall', 'N/A'):.4f}" if metrics.get('recall') else "Recall: N/A")
    
    return model, history, metrics

In [35]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
if __name__ == "__main__":
    model, history, metrics = main()

In [ ]:
#!/usr/bin/env python3

from enhanced_main_adapter import EnhancedAnomalyDetectorWrapper, enhanced_train_and_evaluate_smd, prepare_smd_data

### Dataset2 (TrimSyn)

In [85]:
import torch.optim as optim
import torch.nn as nn
import numpy as np
import pickle as pk
from torch.utils.data import DataLoader, TensorDataset
from torchdiffeq import odeint_adjoint as odeint

from sklearn.metrics import f1_score
import torch.nn.functional as F

In [104]:
# trimSyn 데이터 로드
trimSyn_path = './datasets/MSCRED/trimSyn.pk'
with open(trimSyn_path, 'rb') as pkf:
    data = pk.load(pkf)

x_trn_all = data['x_trn'] 
x_tst_all = data['x_tst']  
lab_tst_all = data['lab_tst']

print(f"트레이닝 데이터 수: {len(x_trn_all)}")
print(f"첫 번째 트레이닝 데이터 형태: {x_trn_all[0].shape}")
print(f"테스트 데이터 수: {len(x_tst_all)}")
for i, x_tst in enumerate(x_tst_all):
    print(f"테스트 섹션 {i} 형태: {x_tst.shape}, 라벨 형태: {lab_tst_all[i].shape}")

트레이닝 데이터 수: 5
첫 번째 트레이닝 데이터 형태: (10000, 30)
테스트 데이터 수: 5
테스트 섹션 0 형태: (1720, 30), 라벨 형태: (1720,)
테스트 섹션 1 형태: (950, 30), 라벨 형태: (950,)
테스트 섹션 2 형태: (3070, 30), 라벨 형태: (3070,)
테스트 섹션 3 형태: (650, 30), 라벨 형태: (650,)
테스트 섹션 4 형태: (1290, 30), 라벨 형태: (1290,)


In [105]:
x_trn_concat = np.stack(x_trn_all, axis=0) 
x_trn_tensor = torch.tensor(x_trn_concat, dtype=torch.float32)

T = x_trn_tensor.shape[1]
time_steps = torch.linspace(0, 1, T) 

input_dim = x_trn_tensor.shape[2]
train_dataset = TensorDataset(x_trn_tensor, time_steps.repeat(x_trn_tensor.shape[0], 1)) 
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [133]:
def train_irregular_ode(model, x_trn_all, x_tst_all, lab_tst_all, num_epochs=30):
    """
    Train and evaluate the irregular ODE model
    
    Args:
        model: The model to train
        x_trn_all: List of training data arrays
        x_tst_all: List of test data arrays
        lab_tst_all: List of test labels
        num_epochs: Number of training epochs
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    # Prepare training data
    print(f"Training data type: {type(x_trn_all)}")
    print(f"Number of training sections: {len(x_trn_all)}")
    
    # Stack all training sections
    x_trn_concat = np.stack(x_trn_all, axis=0)  # (N, T, D)
    x_trn_tensor = torch.tensor(x_trn_concat, dtype=torch.float32)
    
    T = x_trn_tensor.shape[1]
    time_steps = torch.linspace(0, 1, T)
    
    # Create training dataset and loader
    train_dataset = TensorDataset(x_trn_tensor, time_steps.repeat(x_trn_tensor.shape[0], 1))
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    
    # Optimizer setup
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )
    
    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=True
    )
    
    # Training
    model.train()
    history = []
    
    for epoch in range(num_epochs):
        epoch_loss = 0
        epoch_recon_loss = 0
        epoch_kl_loss = 0
        epoch_flow_loss = 0
        
        for batch_idx, (x_batch, times_batch) in enumerate(train_loader):
            x_batch = x_batch.to(device)  # (B, T, D)
            times_batch = times_batch.to(device)  # (B, T)
            
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(x_batch, times_batch)
            losses = outputs["losses"]
            
            # Backward pass
            losses["total_loss"].backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            # Accumulate losses
            epoch_loss += losses["total_loss"].item()
            epoch_recon_loss += losses["recon_loss"].item()
            epoch_kl_loss += losses["kl_loss"].item()
            epoch_flow_loss += losses["flow_loss"].item()
        
        # Average losses per epoch
        n_batches = len(train_loader)
        avg_loss = epoch_loss / n_batches
        avg_recon_loss = epoch_recon_loss / n_batches
        avg_kl_loss = epoch_kl_loss / n_batches
        avg_flow_loss = epoch_flow_loss / n_batches
        
        # Learning rate adjustment
        scheduler.step(avg_loss)
        
        print(f"Epoch {epoch+1}/{num_epochs}, "
              f"Loss: {avg_loss:.4f}, "
              f"Recon: {avg_recon_loss:.4f}, "
              f"KL: {avg_kl_loss:.4f}, "
              f"Flow: {avg_flow_loss:.4f}")
        
        history.append({
            'epoch': epoch + 1,
            'total_loss': avg_loss,
            'recon_loss': avg_recon_loss,
            'kl_loss': avg_kl_loss,
            'flow_loss': avg_flow_loss
        })
    
    # Evaluation
    model.eval()
    all_metrics = []
    
    # Process each test section
    for i, (x_tst, lab_tst) in enumerate(zip(x_tst_all, lab_tst_all)):
        print(f"\nEvaluating test section {i+1}")
        
        # Prepare test data
        x_tst_tensor = torch.tensor(x_tst, dtype=torch.float32)
        if x_tst_tensor.dim() == 2:  # If 2D, add batch dimension
            x_tst_tensor = x_tst_tensor.unsqueeze(0)
        
        T = x_tst_tensor.shape[1]
        time_steps = torch.linspace(0, 1, T)
        time_steps = time_steps.repeat(x_tst_tensor.shape[0], 1)
        
        # Calculate anomaly scores
        with torch.no_grad():
            scores = model.anomaly_score(x_tst_tensor.to(device), time_steps.to(device))
        
        scores = np.array(scores)
        labels = np.array(lab_tst)
        
        # Debug information
        print(f"Test section {i+1} - Scores shape: {scores.shape}, Labels shape: {labels.shape}")
        print(f"Unique labels: {np.unique(labels)}")
        print(f"Scores range: [{scores.min():.4f}, {scores.max():.4f}]")
        
        # Calculate metrics
        try:
            threshold = np.percentile(scores, 90)
            predictions = (scores > threshold).astype(int)
            
            section_metrics = {
                'auc': roc_auc_score(labels, scores) if len(np.unique(labels)) > 1 else None,
                'f1': f1_score(labels, predictions, average='binary'),
                'precision': precision_score(labels, predictions, average='binary', zero_division=0),
                'recall': recall_score(labels, predictions, average='binary', zero_division=0),
                'accuracy': accuracy_score(labels, predictions),
                'threshold': threshold,
                'num_samples': len(labels),
                'num_anomalies': np.sum(labels),
                'num_predicted_anomalies': np.sum(predictions)
            }
            
            print(f"Section {i+1} metrics:")
            for key, value in section_metrics.items():
                if key in ['auc', 'f1', 'precision', 'recall', 'accuracy']:
                    print(f"{key}: {value:.4f}" if value is not None else f"{key}: N/A")
            
        except Exception as e:
            print(f"Error calculating metrics for section {i+1}: {e}")
            section_metrics = {
                'auc': None,
                'f1': 0.0,
                'precision': 0.0,
                'recall': 0.0,
                'accuracy': 0.0
            }
        
        all_metrics.append(section_metrics)
    
    # Calculate average metrics
    avg_metrics = {}
    for metric in ['accuracy', 'f1', 'precision', 'recall']:
        valid_values = [m[metric] for m in all_metrics if m[metric] is not None]
        if valid_values:
            avg_metrics[metric] = np.mean(valid_values)
        else:
            avg_metrics[metric] = None
    
    valid_auc = [m['auc'] for m in all_metrics if m['auc'] is not None]
    if valid_auc:
        avg_metrics['auc'] = np.mean(valid_auc)
    else:
        avg_metrics['auc'] = None
    
    print("\nOverall Performance:")
    for key, value in avg_metrics.items():
        if value is not None:
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: N/A")
    
    return model, history, {'all_metrics': all_metrics, 'avg_metrics': avg_metrics}

In [134]:
def prepare_irregular_data(x_data, timestamps=None, missing_ratio=0.3, random_seed=42):

    np.random.seed(random_seed)
    N, T, D = x_data.shape
    
    if timestamps is None:
        base_times = np.linspace(0, 1, T)
        timestamps = np.tile(base_times, (N, 1)) 
    
    mask = np.ones((N, T))
    
    for i in range(N):
        missing_idx = np.random.choice(
            np.arange(1, T), 
            size=int(missing_ratio * (T-1)), 
            replace=False
        )
        mask[i, missing_idx] = 0
    
    x_tensor = torch.tensor(x_data, dtype=torch.float32)
    timestamps_tensor = torch.tensor(timestamps, dtype=torch.float32)
    mask_tensor = torch.tensor(mask, dtype=torch.float32)
    
    return x_tensor, timestamps_tensor, mask_tensor

In [135]:
def evaluate_anomaly_detector(
    model, 
    test_data=None, 
    test_labels=None, 
    test_loader=None,
    device=None, 
    threshold=None,
    threshold_method='auto',
    threshold_percentile=95,
    return_scores=False,
    verbose=True,
    is_section_eval=False
):

    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    
    if test_loader is not None and (test_data is not None or test_labels is not None):
        raise ValueError("Cannot use test_loader with test_data/test_labels")
    if test_loader is None and (test_data is None or test_labels is None):
        raise ValueError("Must provide either test_loader or test_data/test_labels")
    
    if is_section_eval:
        all_metrics = []
        for i, (section_data, section_labels) in enumerate(zip(test_data, test_labels)):
            if verbose:
                print(f"\nSection {i+1} evaluation")
                print(f"Data shape: {section_data.shape}, Labels shape: {section_labels.shape}")
            
            try:
                if len(section_data) > 100:
                    val_size = min(100, len(section_data) // 5)
                    val_data, val_labels = section_data[:val_size], section_labels[:val_size]
                    best_threshold, best_f1, _, _ = find_optimal_threshold(model, val_data, val_labels, device)
                    if verbose:
                        print(f"Optimal threshold: {best_threshold:.4f}, Validation F1: {best_f1:.4f}")
                else:
                    best_threshold = None
                
                section_metrics = evaluate_anomaly_detector(
                    model, section_data, section_labels,
                    threshold=best_threshold,
                    device=device,
                    verbose=verbose,
                    is_section_eval=False
                )
                all_metrics.append(section_metrics)
                
            except Exception as e:
                if verbose:
                    print(f"Error in section {i+1}: {e}")
                all_metrics.append({
                    'accuracy': None, 'f1': None, 'precision': None,
                    'recall': None, 'auc': None
                })
        
        if verbose:
            print("\nOverall Performance")
        
        avg_metrics = {}
        for metric in ['accuracy', 'f1', 'precision', 'recall']:
            valid_values = [m[metric] for m in all_metrics if m[metric] is not None]
            avg_metrics[metric] = np.mean(valid_values) if valid_values else None
        
        valid_auc = [m['auc'] for m in all_metrics if 'auc' in m and m['auc'] is not None]
        avg_metrics['auc'] = np.mean(valid_auc) if valid_auc else None
        
        if verbose:
            for key, value in avg_metrics.items():
                if value is not None and not np.isnan(value):
                    print(f"{key}: {value:.4f}")
                else:
                    print(f"{key}: Not available")
        
        return {'all_metrics': all_metrics, 'avg_metrics': avg_metrics}
    
    anomaly_scores = []
    all_labels = []
    
    with torch.no_grad():
        if test_loader is not None:
            for batch_data in test_loader:
                if len(batch_data) == 3:
                    x_batch, times_batch, mask_batch = [b.to(device) for b in batch_data]
                    scores = model.anomaly_score(x_batch, times_batch, mask_batch)
                    anomaly_scores.extend(scores)
                else:
                    raise ValueError("Unexpected batch format")
            
            if test_labels is None:
                raise ValueError("test_labels required with test_loader")
            all_labels = test_labels
            
        else:
            if isinstance(test_data, np.ndarray):
                test_data = torch.tensor(test_data, dtype=torch.float32)
            
            if test_data.dim() == 2:
                test_data = test_data.unsqueeze(0)
            
            T = test_data.shape[1]
            timestamps = torch.linspace(0, 1, T).unsqueeze(0).expand(test_data.shape[0], -1).to(device)
            mask = torch.ones_like(timestamps)
            
            scores = model.anomaly_score(test_data.to(device), timestamps, mask)
            anomaly_scores = scores
            all_labels = test_labels
    
    anomaly_scores = np.array(anomaly_scores)
    all_labels = np.array(all_labels)
    
    if verbose:
        print(f"Shapes - Scores: {anomaly_scores.shape}, Labels: {all_labels.shape}")
    
    if anomaly_scores.ndim > 1:
        scores_flat = anomaly_scores.squeeze() if anomaly_scores.shape[0] == 1 else np.max(anomaly_scores, axis=1)
    else:
        scores_flat = anomaly_scores
    
    labels_flat = all_labels.reshape(-1)
    
    if threshold is None:
        if threshold_method == 'percentile':
            threshold = np.percentile(scores_flat, threshold_percentile)
        elif threshold_method == 'normal_based':
            normal_scores = scores_flat[labels_flat == 0]
            threshold = (np.mean(normal_scores) + 2 * np.std(normal_scores)) if len(normal_scores) > 0 else np.percentile(scores_flat, 95)
        elif threshold_method == 'auto':
            normal_scores = scores_flat[labels_flat == 0]
            threshold = (np.mean(normal_scores) + 2 * np.std(normal_scores)) if len(normal_scores) > len(scores_flat) * 0.7 else np.percentile(scores_flat, threshold_percentile)
        else:
            raise ValueError(f"Unsupported threshold_method: {threshold_method}")
    
    predictions = (scores_flat > threshold).astype(int)
    
    metrics = {}
    try:
        metrics.update({
            'accuracy': accuracy_score(labels_flat, predictions),
            'precision': precision_score(labels_flat, predictions, average='binary', zero_division=0),
            'recall': recall_score(labels_flat, predictions, average='binary', zero_division=0),
            'f1': f1_score(labels_flat, predictions, average='binary', zero_division=0),
            'auc': roc_auc_score(labels_flat, scores_flat) if len(np.unique(labels_flat)) > 1 else None
        })
    except Exception as e:
        if verbose:
            print(f"Error calculating metrics: {e}")
        metrics.update({
            'accuracy': 0.0, 'precision': 0.0, 'recall': 0.0,
            'f1': 0.0, 'auc': None
        })
    
    metrics.update({
        'threshold': threshold,
        'threshold_method': threshold_method,
        'num_samples': len(labels_flat),
        'num_anomalies': np.sum(labels_flat),
        'num_predicted_anomalies': np.sum(predictions)
    })
    
    if return_scores:
        metrics.update({
            'anomaly_scores': scores_flat,
            'predictions': predictions,
            'labels': labels_flat
        })
    
    if verbose:
        print(f"\nEvaluation Results:")
        print(f"Threshold ({threshold_method}): {threshold:.4f}")
        print(f"Accuracy: {metrics['accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall: {metrics['recall']:.4f}")
        print(f"F1 Score: {metrics['f1']:.4f}")
        if metrics['auc'] is not None:
            print(f"AUC: {metrics['auc']:.4f}")
        print(f"Samples: {metrics['num_samples']}, "
              f"True Anomalies: {metrics['num_anomalies']}, "
              f"Predicted: {metrics['num_predicted_anomalies']}")
    
    return metrics

In [138]:
def train_irregular_ode(model, train_loader, optimizer, scheduler, num_epochs=30):
    """
    Irregular Time Series ODE 모델 학습
    """
    model.train()
    history = []
    
    for epoch in range(num_epochs):
        epoch_loss = 0
        epoch_recon_loss = 0
        epoch_kl_loss = 0
        epoch_flow_loss = 0
        
        for batch_idx, (x_batch, times_batch, mask_batch) in enumerate(train_loader):
            x_batch = x_batch.to(device)
            times_batch = times_batch.to(device)
            mask_batch = mask_batch.to(device)
            
            # 디버그 정보 출력 (첫 번째 배치, 첫 번째 에폭)
            if batch_idx == 0 and epoch == 0:
                print(f"시간 텐서 형태: {times_batch.shape}")
                print(f"시간 텐서 값 (첫 배치의 첫 시퀀스): {times_batch[0]}")
                print(f"시간이 단조 증가하는가: {(times_batch[0, 1:] > times_batch[0, :-1]).all()}")
                print(f"마스크 텐서 형태: {mask_batch.shape}")
                print(f"마스크 비율: {mask_batch.mean().item():.2f}")
            
            optimizer.zero_grad()
            
            # 순전파
            outputs = model(x_batch, times_batch, mask_batch)
            losses = outputs["losses"]
            
            # 역전파
            losses["total_loss"].backward()
            
            # 그라디언트 클리핑
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
            
            optimizer.step()
            
            # 손실 누적
            epoch_loss += losses["total_loss"].item()
            epoch_recon_loss += losses["recon_loss"].item()
            epoch_kl_loss += losses["kl_loss"].item()
            epoch_flow_loss += losses["flow_loss"].item()
        
        # 에폭별 평균 손실
        n_batches = len(train_loader)
        avg_loss = epoch_loss / n_batches
        avg_recon_loss = epoch_recon_loss / n_batches
        avg_kl_loss = epoch_kl_loss / n_batches
        avg_flow_loss = epoch_flow_loss / n_batches
        
        # 로그 출력
        print(f"Epoch {epoch+1}/{num_epochs}, "
              f"Loss: {avg_loss:.4f}, "
              f"Recon: {avg_recon_loss:.4f}, "
              f"KL: {avg_kl_loss:.4f}, "
              f"Flow: {avg_flow_loss:.4f}")
        
        # 학습률 조정
        scheduler.step(avg_loss)
        
        # 손실 기록
        history.append({
            'epoch': epoch + 1,
            'total_loss': avg_loss,
            'recon_loss': avg_recon_loss,
            'kl_loss': avg_kl_loss,
            'flow_loss': avg_flow_loss
        })
    
    return model, history

In [139]:
def train_irregular_ode(model, train_loader, optimizer, scheduler, num_epochs=5):
    model.train()
    history = []
    
    for epoch in range(num_epochs):
        epoch_loss = 0
        epoch_recon_loss = 0
        epoch_kl_loss = 0
        epoch_flow_loss = 0
        
        for batch_idx, (x_batch, times_batch, mask_batch) in enumerate(train_loader):
            x_batch = x_batch.to(device)
            times_batch = times_batch.to(device)
            mask_batch = mask_batch.to(device)
            
            optimizer.zero_grad()
            
            # 모델 순전파 (결측값 마스크 포함)
            outputs = model(x_batch, times_batch, mask_batch)
            losses = outputs["losses"]
            
            # 역전파
            losses["total_loss"].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
            optimizer.step()
            
            # 손실 누적
            epoch_loss += losses["total_loss"].item()
            epoch_recon_loss += losses["recon_loss"].item()
            epoch_kl_loss += losses["kl_loss"].item()
            epoch_flow_loss += losses["flow_loss"].item()
        
        # 에폭별 평균 손실
        avg_loss = epoch_loss / len(train_loader)
        avg_recon_loss = epoch_recon_loss / len(train_loader)
        avg_kl_loss = epoch_kl_loss / len(train_loader)
        avg_flow_loss = epoch_flow_loss / len(train_loader)
        
        # 학습률 조정
        scheduler.step(avg_loss)
        
        print(f"Epoch {epoch+1}/{num_epochs}, "
              f"Loss: {avg_loss:.4f}, "
              f"Recon: {avg_recon_loss:.4f}, "
              f"KL: {avg_kl_loss:.4f}, "
              f"Flow: {avg_flow_loss:.4f}")
        
        # 손실 기록
        history.append({
            'epoch': epoch + 1,
            'total_loss': avg_loss,
            'recon_loss': avg_recon_loss,
            'kl_loss': avg_kl_loss,
            'flow_loss': avg_flow_loss
        })
    
    return model, history

In [ ]:
x_tensor, timestamps, mask = prepare_irregular_data(
    x_trn_concat, 
    missing_ratio=0.1 
)

# 데이터셋 및 데이터로더 생성
irregular_dataset = TensorDataset(x_tensor, timestamps, mask)
irregular_loader = DataLoader(irregular_dataset, batch_size=32, shuffle=True)

# 모델 초기화
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = IrregularTimeSeriesAnomalyDetector(
    input_dim=x_tensor.shape[2],
    latent_dim=32,
    hidden_dim=64,
    flow_layers=4,
    beta=0.1
).to(device)

# 옵티마이저
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, verbose=True
)


# 모델 훈련
model, history = train_irregular_ode(
    model, irregular_loader, optimizer, scheduler, num_epochs=10
)

Time2Vec initialized with:
- Total output dimension: 8
- Linear component dimension: 1
- Periodic component dimension: 7
Time2Vec initialized with:
- Total output dimension: 8
- Linear component dimension: 1
- Periodic component dimension: 7


/opt/anaconda3/envs/research/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1/10, Loss: 1.7208, Recon: 0.9355, KL: 0.6168, Flow: 17.1043
Epoch 2/10, Loss: 1.8364, Recon: 0.9462, KL: 0.6085, Flow: 16.9993


In [ ]:
def evaluate_irregular_model(model, test_data, test_labels, threshold=None, device=None):
    """
    불규칙 시계열 모델 평가 (차원 처리 개선)
    """
    if device is None:
        device = next(model.parameters()).device
    
    model.eval()
    
    # 테스트 데이터 준비
    if isinstance(test_data, np.ndarray):
        test_data = torch.tensor(test_data, dtype=torch.float32)
    
    # 배치 차원 확인 및 추가
    if test_data.dim() == 2:  # (T, D) 형태인 경우
        test_data = test_data.unsqueeze(0)  # (1, T, D) 형태로 변환
    
    # 시간 벡터 생성
    T = test_data.shape[1]
    timestamps = torch.linspace(0, 1, T).unsqueeze(0).to(device)
    
    # 전체 마스크 (테스트에서는 모든 데이터 사용)
    mask = torch.ones_like(timestamps)
    
    # 데이터를 디바이스로 이동
    test_data = test_data.to(device)
    timestamps = timestamps.to(device)
    mask = mask.to(device)
    
    # 이상치 점수 계산
    with torch.no_grad():
        anomaly_scores = model.anomaly_score(test_data, timestamps, mask)
    
    # 점수와 라벨 형태 맞추기
    if test_labels.ndim == 1 and test_data.shape[0] == 1:  
        # 단일 시퀀스 (1, T, D)와 벡터 라벨 (T,)
        scores_flat = anomaly_scores.squeeze()  # (T,)
        labels_flat = test_labels
    elif test_labels.ndim == 2:  
        # 다중 시퀀스 (B, T, D)와 2D 라벨 (B, T)
        scores_flat = anomaly_scores.reshape(-1)  # (B*T,)
        labels_flat = test_labels.reshape(-1)
    else:  
        # 다중 시퀀스 (B, T, D)와 1D 라벨 (B,)
        scores_flat = np.max(anomaly_scores, axis=1)  # (B,)
        labels_flat = test_labels
    
    # 임계값을 지정하지 않은 경우 자동 계산
    if threshold is None:
        # 정상 데이터의 평균 + 2 * 표준편차
        normal_scores = scores_flat[labels_flat == 0]
        if len(normal_scores) > 0:
            threshold = np.mean(normal_scores) + 2 * np.std(normal_scores)
        else:
            threshold = np.percentile(scores_flat, 95)  # 상위 5%를 이상치로 판단
    
    # 이상치 예측
    y_pred = (scores_flat > threshold).astype(int)
    
    # 성능 지표 계산
    metrics = {
        'accuracy': accuracy_score(labels_flat, y_pred),
        'f1': f1_score(labels_flat, y_pred, average='macro'),
        'precision': precision_score(labels_flat, y_pred, average='macro', zero_division=0),
        'recall': recall_score(labels_flat, y_pred, average='macro', zero_division=0),
    }
    
    # ROC-AUC (레이블이 모두 같은 클래스가 아닌 경우에만)
    if len(np.unique(labels_flat)) > 1:
        metrics['auc'] = roc_auc_score(labels_flat, scores_flat)
    
    # 임계값 저장
    metrics['threshold'] = threshold
    
    return metrics

def find_optimal_threshold(model, validation_data, validation_labels, device=None, n_thresholds=100):
    """
    최적의 임계값 탐색 (데이터 형태 처리 개선)
    """
    if device is None:
        device = next(model.parameters()).device
    
    model.eval()
    
    # 데이터 준비 및 차원 확인
    if isinstance(validation_data, np.ndarray):
        validation_data = torch.tensor(validation_data, dtype=torch.float32)
    
    # 배치 차원 확인 및 추가
    if validation_data.dim() == 2:  # (T, D) 형태인 경우
        validation_data = validation_data.unsqueeze(0)  # (1, T, D) 형태로 변환
    
    # 차원 정보 출력 (디버깅 용도)
    print(f"검증 데이터 형태: {validation_data.shape}")
    print(f"검증 라벨 형태: {validation_labels.shape}")
    
    # 시간 벡터 및 마스크 생성
    T = validation_data.shape[1]
    timestamps = torch.linspace(0, 1, T).unsqueeze(0).to(device)
    mask = torch.ones_like(timestamps)
    
    # 디바이스 이동
    validation_data = validation_data.to(device)
    timestamps = timestamps.to(device)
    mask = mask.to(device)
    
    # 이상치 점수 계산
    with torch.no_grad():
        anomaly_scores = model.anomaly_score(validation_data, timestamps, mask)
    
    print(f"이상치 점수 형태: {anomaly_scores.shape}")
    
    # 점수와 라벨 형태 맞추기
    if validation_labels.ndim == 1 and validation_data.shape[0] == 1:  
        # 단일 시퀀스 (1, T, D)와 벡터 라벨 (T,)
        scores_flat = anomaly_scores.squeeze()  # (T,) 형태로 변환
        labels_flat = validation_labels
    elif validation_labels.ndim == 2:  
        # 다중 시퀀스 (B, T, D)와 2D 라벨 (B, T)
        scores_flat = anomaly_scores.reshape(-1)  # (B*T,) 형태로 변환
        labels_flat = validation_labels.reshape(-1)
    else:  
        # 다중 시퀀스 (B, T, D)와 1D 라벨 (B,) → 시퀀스별 최대값
        scores_flat = np.max(anomaly_scores, axis=1)  # (B,) 형태
        labels_flat = validation_labels
    
    print(f"변환 후 점수 형태: {scores_flat.shape}")
    print(f"변환 후 라벨 형태: {labels_flat.shape}")
    
    # 임계값 탐색
    min_score = np.min(scores_flat)
    max_score = np.max(scores_flat)
    thresholds = np.linspace(min_score, max_score, n_thresholds)
    
    f1_scores = []
    for threshold in thresholds:
        y_pred = (scores_flat > threshold).astype(int)
        f1 = f1_score(labels_flat, y_pred, average='macro')
        f1_scores.append(f1)
    
    # 최적 임계값 선택
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]
    
    return best_threshold, best_f1, thresholds, f1_scores

In [ ]:
def prepare_test_data(x_tst_all, lab_tst_all):
    """테스트 데이터와 라벨 준비 (형태 정보 출력)"""
    test_data_list = []
    test_labels_list = []
    
    for i, (x_tst, lab_tst) in enumerate(zip(x_tst_all, lab_tst_all)):
        # 텐서로 변환
        x_tst_tensor = torch.tensor(x_tst, dtype=torch.float32)
        print(f"섹션 {i+1} - 데이터 형태: {x_tst_tensor.shape}, 라벨 형태: {lab_tst.shape}")
        test_data_list.append(x_tst_tensor)
        test_labels_list.append(lab_tst)
    
    return test_data_list, test_labels_list

test_data_list, test_labels_list = prepare_test_data(x_tst_all, lab_tst_all)

all_metrics = []

for i, (test_data, test_labels) in enumerate(zip(test_data_list, test_labels_list)):
    print(f"\n===== 섹션 {i+1} 평가 =====")
    print(f"테스트 데이터 형태: {test_data.shape}")
    print(f"테스트 라벨 형태: {test_labels.shape}")
    
    # 검증 데이터 준비
    if len(test_data) > 100:
        val_size = min(100, len(test_data) // 5)
        
        # 여기서 데이터 형태 확인 및 조정이 필요
        if len(test_data.shape) == 3:  # (B, T, D) 형태
            val_data = test_data[:val_size]
            val_labels = test_labels[:val_size]
        else:  # (T, D) 형태
            val_data = test_data[:val_size]
            if len(test_labels.shape) == 1:  # (T,) 형태
                val_labels = test_labels[:val_size]
            else:
                val_labels = test_labels[:val_size, :]
        
        print(f"검증 데이터 크기: {val_data.shape}")
        print(f"검증 라벨 크기: {val_labels.shape if hasattr(val_labels, 'shape') else len(val_labels)}")
        
        # 임계값 찾기
        best_threshold, best_f1, _, _ = find_optimal_threshold(
            model, val_data, val_labels, device
        )
        print(f"최적 임계값: {best_threshold:.4f}, 검증 F1: {best_f1:.4f}")
    else:
        best_threshold = None
    
    # 섹션별 테스트 평가
    metrics = evaluate_irregular_model(
        model, test_data, test_labels, threshold=best_threshold, device=device
    )
    
    # 결과 출력
    print("\n성능 지표:")
    for key, value in metrics.items():
        if isinstance(value, float):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")
    
    # 결과 저장
    metrics['section'] = i+1
    all_metrics.append(metrics)
# 평균 성능 계산
print("\n===== All mean performance =====")
avg_metrics = {
    'accuracy': np.mean([m['accuracy'] for m in all_metrics]),
    'f1': np.mean([m['f1'] for m in all_metrics]),
    'precision': np.mean([m['precision'] for m in all_metrics]),
    'recall': np.mean([m['recall'] for m in all_metrics])
}

if all('auc' in m for m in all_metrics):
    avg_metrics['auc'] = np.mean([m['auc'] for m in all_metrics])

for key, value in avg_metrics.items():
    print(f"{key}: {value:.4f}")
    


def visualize_anomaly_detection(model, test_data, test_labels, threshold=None, sample_idx=0, max_features=4):
    """이상치 탐지 결과 시각화 (차원 처리 개선)"""    
    model.eval()
    
    # 데이터 준비
    if isinstance(test_data, np.ndarray):
        test_data = torch.tensor(test_data, dtype=torch.float32)
    
    # 차원 확인 및 출력
    print(f"시각화 데이터 형태: {test_data.shape}")
    print(f"시각화 라벨 형태: {test_labels.shape if hasattr(test_labels, 'shape') else len(test_labels)}")
    
    # 적절한 샘플 선택
    if len(test_data.shape) == 3:  # (B, T, D) 형태
        if sample_idx < test_data.shape[0]:
            x_sample = test_data[sample_idx:sample_idx+1].to(device)  
        else:
            print(f"샘플 인덱스 {sample_idx}가 범위를 벗어남. 첫 번째 샘플 사용.")
            x_sample = test_data[0:1].to(device)  
    else: 
        x_sample = test_data.unsqueeze(0).to(device)  
    
    # 시간 벡터 생성
    T = x_sample.shape[1]
    timestamps = torch.linspace(0, 1, T).unsqueeze(0).to(device)
    mask = torch.ones_like(timestamps).to(device)
    
    # 재구성 및 이상치 점수 계산
    with torch.no_grad():
        outputs = model(x_sample, timestamps, mask)
        recon_mu = outputs['recon_mu'].cpu().numpy()[0]
        recon_std = torch.exp(0.5 * outputs['recon_logvar']).cpu().numpy()[0]
        anomaly_scores = model.anomaly_score(x_sample, timestamps, mask)[0]
    
    # 원본 데이터
    x_np = x_sample.cpu().numpy()[0]
    
    # 라벨 처리
    if hasattr(test_labels, 'shape'):
        if len(test_labels.shape) == 2:  # (B, T) 형태
            y_true = test_labels[sample_idx] if sample_idx < len(test_labels) else test_labels[0]
        elif len(test_labels.shape) == 1 and len(test_data.shape) == 3:  # (B,) 형태, 데이터는 (B, T, D)
            # 시퀀스별 라벨 - 전체 시퀀스에 동일 라벨 적용
            y_true = test_labels[sample_idx] * np.ones(T) if sample_idx < len(test_labels) else test_labels[0] * np.ones(T)
        else:  # (T,) 형태, 데이터는 (T, D) 또는 (1, T, D)
            y_true = test_labels
    else:  # 리스트 등의 형태
        y_true = np.array(test_labels)
        if len(y_true.shape) == 1 and len(y_true) != T:
            y_true = np.zeros(T)  
    
    # 임계값 설정
    if threshold is None:
        normal_scores = anomaly_scores[y_true == 0]
        if len(normal_scores) > 0:
            threshold = np.mean(normal_scores) + 2 * np.std(normal_scores)
        else:
            threshold = np.percentile(anomaly_scores, 95)
    
    # 이상치 예측
    y_pred = (anomaly_scores > threshold).astype(int)
    
    # 시각화
    feature_count = min(max_features, x_np.shape[1])
    time_points = np.linspace(0, 1, T)
    
    fig, axes = plt.subplots(feature_count + 1, 1, figsize=(14, 3 * (feature_count + 1)))
    
    # 이상치 점수 시각화
    ax = axes[0]
    ax.plot(time_points, anomaly_scores, 'b-', label='Anomaly Score')
    ax.axhline(y=threshold, color='r', linestyle='--', label=f'Threshold: {threshold:.4f}')
    
    # 실제 이상치 영역 표시
    anomaly_regions = np.where(y_true == 1)[0]
    for idx in anomaly_regions:
        if idx < len(time_points):  
            ax.axvspan(time_points[idx] - 0.01, time_points[idx] + 0.01, alpha=0.2, color='red')
    
    # 탐지된 이상치 표시
    detected = np.where(y_pred == 1)[0]
    ax.scatter(time_points[detected], anomaly_scores[detected], color='red', marker='x', s=50, label='Detected Anomaly')
    
    ax.set_title('Anomaly Score')
    ax.set_xlabel('Time')
    ax.set_ylabel('Score')
    ax.legend()
    
    # 주요 특성별 시각화
    for i in range(feature_count):
        ax = axes[i + 1]
        
        # 원본 데이터
        ax.plot(time_points, x_np[:, i], 'b-', label='Original')
        
        # 재구성 데이터 (불확실성 포함)
        ax.plot(time_points, recon_mu[:, i], 'g-', label='Reconstruction')
        ax.fill_between(
            time_points,
            recon_mu[:, i] - 2 * recon_std[:, i],
            recon_mu[:, i] + 2 * recon_std[:, i],
            color='g', alpha=0.2
        )
        
        # 이상치 영역 표시
        for idx in anomaly_regions:
            if idx < len(time_points): 
                ax.axvspan(time_points[idx] - 0.01, time_points[idx] + 0.01, alpha=0.2, color='red')
        
        # 탐지된 이상치 표시
        for idx in detected:
            if idx < len(time_points) and idx < x_np.shape[0]: 
                ax.scatter(time_points[idx], x_np[idx, i], color='red', marker='x', s=50)
        
        ax.set_title(f'Section {i+1}')
        ax.set_xlabel('Time')
        ax.set_ylabel('Value')
        ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    # 정확도 계산 (라벨과 예측이 동일한 형태인 경우만)
    if len(y_true) == len(y_pred):
        accuracy = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, average='macro')
        precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
        recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
        
        print(f"Sample {sample_idx} Performance:")
        print(f"Accuracy: {accuracy:.4f}, F1: {f1:.4f}")
        print(f"Precision: {precision:.4f}, Recall: {recall:.4f}")
    else:
        print(f"라벨 형태({len(y_true)})와 예측 형태({len(y_pred)})가 일치하지 않아 성능 계산 불가")
    
    return anomaly_scores, y_pred, y_true

# 샘플 시각화
for section_idx in range(len(test_data_list)):
    print(f"\n===== Section {section_idx+1} Visualization =====")
    sample_idx = 0  # 첫 번째 샘플
    
    # 섹션별 최적 임계값 사용
    threshold = all_metrics[section_idx]['threshold']
    
    # 시각화
    visualize_anomaly_detection(
        model,
        test_data_list[section_idx], 
        test_labels_list[section_idx],
        threshold=threshold,
        sample_idx=sample_idx
    )

In [ ]:
from datasets.SMD.preprocess_SMD import SMD_Dataset
import pickle as pk

trimSyn_path = './data/MSCRED/trimSyn.pk'
with open(trimSyn_path, 'rb') as pkf:
    trimSyn_data = pk.load(pkf)
    
results = train_anomaly_detector(
    data=trimSyn_data,
    dataset_name='trimSyn',
    num_epochs=30,
    batch_size=1024
)

### Dataset 3 (SWaT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

In [ ]:
!unzip ./datasets/SWaT/SWaT_Dataset_v0.zip -d ./datasets/SWaT/

In [ ]:
swat_normal = pd.read_csv("./datasets/SWaT/SWaT_Dataset_Normal_v0.csv", skiprows=1)
swat_attack = pd.read_csv("./datasets/SWaT/SWaT_Dataset_Attack_v0.csv")

swat_normal.columns = swat_normal.columns.str.replace(' ','')
swat_attack.columns = swat_attack.columns.str.replace(' ','')

In [ ]:
swat_attack["Normal/Attack"].replace(' ', '', regex=True, inplace=True)

In [ ]:
num_att = len(swat_attack[swat_attack["Normal/Attack"] == "Attack"])*100/ (len(swat_normal)+len(swat_attack))
print(f'there are {np.round(num_att, 2)} % of attack data')

### Dataset 4 (WADI)

In [ ]:
from datetime import datetime

In [ ]:
normal = pd.read_csv("./datasets/WADI/WADI_14days.csv",sep=',', skiprows=[0,1,2,3],skip_blank_lines=True)
attack = pd.read_csv("./datasets/WADI/WADI_attackdata.csv",sep=",")

In [ ]:
labels=[]

attack.reset_index()
for index, row in attack.iterrows():
    date_temp=row['Date']
    date_mask="%m/%d/%Y"
    date_obj=datetime.strptime(date_temp, date_mask)
    time_temp=row['Time']
    time_mask="%I:%M:%S.%f %p"
    time_obj=datetime.strptime(time_temp,time_mask)

    if date_obj==datetime.strptime('10/9/2017', '%m/%d/%Y'):
        if time_obj>=datetime.strptime('7:25:00.000 PM', '%I:%M:%S.%f %p') and time_obj<=datetime.strptime('7:50:16.000 PM', '%I:%M:%S.%f %p'):
            labels.append('Attack')
            continue

    if date_obj==datetime.strptime('10/10/2017', '%m/%d/%Y'):
        if time_obj>=datetime.strptime('10:24:10.000 AM', '%I:%M:%S.%f %p') and time_obj<=datetime.strptime('10:34:00.000 AM', '%I:%M:%S.%f %p'):
            labels.append('Attack')
            continue
        elif time_obj>=datetime.strptime('10:55:00.000 AM', '%I:%M:%S.%f %p') and time_obj<=datetime.strptime('11:24:00.000 AM', '%I:%M:%S.%f %p'):
            labels.append('Attack')
            continue
        elif time_obj>=datetime.strptime('11:30:40.000 AM', '%I:%M:%S.%f %p') and time_obj<=datetime.strptime('11:44:50.000 AM', '%I:%M:%S.%f %p'):
            labels.append('Attack')
            continue
        elif time_obj>=datetime.strptime('1:39:30.000 PM', '%I:%M:%S.%f %p') and time_obj<=datetime.strptime('1:50:40.000 PM', '%I:%M:%S.%f %p'):
            labels.append('Attack')
            continue
        elif time_obj>=datetime.strptime('2:48:17.000 PM', '%I:%M:%S.%f %p') and time_obj<=datetime.strptime('2:59:55.000 PM', '%I:%M:%S.%f %p'):
            labels.append('Attack')
            continue
        elif time_obj>=datetime.strptime('5:40:00.000 PM', '%I:%M:%S.%f %p') and time_obj<=datetime.strptime('5:49:40.000 PM', '%I:%M:%S.%f %p'):
            labels.append('Attack')
            continue
        elif time_obj>=datetime.strptime('10:55:00.000 AM', '%I:%M:%S.%f %p') and time_obj<=datetime.strptime('10:56:27.000 AM', '%I:%M:%S.%f %p'):
            labels.append('Attack')
            continue
    
    if date_obj==datetime.strptime('10/11/2017', '%m/%d/%Y'):
        if time_obj>=datetime.strptime('11:17:54.000 AM', '%I:%M:%S.%f %p') and time_obj<=datetime.strptime('11:31:20.000 AM', '%I:%M:%S.%f %p'):
            labels.append('Attack')
            continue
        elif time_obj>=datetime.strptime('11:36:31.000 AM', '%I:%M:%S.%f %p') and time_obj<=datetime.strptime('11:47:00.000 AM', '%I:%M:%S.%f %p'):
            labels.append('Attack')
            continue
        elif time_obj>=datetime.strptime('11:59:00.000 AM', '%I:%M:%S.%f %p') and time_obj<=datetime.strptime('12:05:00.000 PM', '%I:%M:%S.%f %p'):
            labels.append('Attack')
            continue
        elif time_obj>=datetime.strptime('12:07:30.000 PM', '%I:%M:%S.%f %p') and time_obj<=datetime.strptime('12:10:52.000 PM', '%I:%M:%S.%f %p'):
            labels.append('Attack')
            continue
        elif time_obj>=datetime.strptime('12:16:00.000 PM', '%I:%M:%S.%f %p') and time_obj<=datetime.strptime('12:25:36.000 PM', '%I:%M:%S.%f %p'):
            labels.append('Attack')
            continue
        elif time_obj>=datetime.strptime('3:26:30.000 PM', '%I:%M:%S.%f %p') and time_obj<=datetime.strptime('3:37:00.000 PM', '%I:%M:%S.%f %p'):
            labels.append('Attack')
            continue

    labels.append('Normal')
        